# 01 — Exploration

Sanity checks on the warehouse before doing any real analysis: row counts, null rates on the
columns the business questions depend on, and date-range coverage. Connects via SQLAlchemy to
the same BigQuery project the dashboard reads from — no separate data copy.

See `docs/architecture/README.md` for what each dataset holds:
- `olist_raw` — one table per source CSV, near-verbatim (dlt-managed, not queried directly below)
- `olist_warehouse` — staging views + star schema (`fact_orders`, `dim_customer`, `dim_date`)
- `olist_reporting` — the pre-aggregated datamart the dashboard reads

In [1]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

REPO_ROOT = Path.cwd().parent
load_dotenv(REPO_ROOT / "config" / "credentials.env")

PROJECT_ID = os.environ["GOOGLE_CLOUD_PROJECT"]
engine = create_engine(f"bigquery://{PROJECT_ID}")

def q(sql: str) -> pd.DataFrame:
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)

PROJECT_ID

'ntu-bigdata-project'

## Row counts

One row per table across all three datasets — first thing to check after any pipeline run.

In [2]:
tables = {
    "olist_warehouse": ["stg_orders", "stg_order_items", "stg_order_reviews", "stg_customers",
                         "int_order_delivery_stages", "dim_customer", "dim_date", "fact_orders"],
    "olist_reporting": ["mart_delivery_kpis", "mart_satisfaction_by_delivery", "mart_monthly_sales"],
}

counts = []
for dataset, table_list in tables.items():
    for table in table_list:
        n = q(f"SELECT COUNT(*) AS n FROM `{PROJECT_ID}.{dataset}.{table}`")["n"].iloc[0]
        counts.append({"dataset": dataset, "table": table, "row_count": n})

row_counts = pd.DataFrame(counts)
row_counts

,dataset,table,row_count
0,olist_warehouse,stg_orders,99441
1,olist_warehouse,stg_order_items,112650
2,olist_warehouse,stg_order_reviews,99224
3,olist_warehouse,stg_customers,99441
4,olist_warehouse,int_order_delivery_stages,99441
5,olist_warehouse,dim_customer,99441
6,olist_warehouse,dim_date,1096
7,olist_warehouse,fact_orders,99441
8,olist_reporting,mart_delivery_kpis,565
9,olist_reporting,mart_satisfaction_by_delivery,6


## Null checks on `fact_orders`

The columns the delay-math and satisfaction analysis depend on. Some nulls are expected here (e.g. undelivered orders have no `order_delivered_customer_at`) — this is about knowing the *rate*, not assuming zero.

In [3]:
null_check_sql = """
SELECT
    COUNT(*) AS total_rows,
    COUNTIF(order_purchase_at IS NULL) AS null_purchase_at,
    COUNTIF(order_approved_at IS NULL) AS null_approved_at,
    COUNTIF(order_delivered_carrier_at IS NULL) AS null_delivered_carrier_at,
    COUNTIF(order_delivered_customer_at IS NULL) AS null_delivered_customer_at,
    COUNTIF(order_estimated_delivery_at IS NULL) AS null_estimated_delivery_at,
    COUNTIF(avg_review_score IS NULL) AS null_avg_review_score,
    COUNTIF(delivery_status IS NULL) AS null_delivery_status
FROM `{project}.olist_warehouse.fact_orders`
""".format(project=PROJECT_ID)

null_counts = q(null_check_sql)
null_pct = null_counts.T.rename(columns={0: "count"})
null_pct["pct_of_total"] = (null_pct["count"] / null_counts["total_rows"].iloc[0] * 100).round(1)
null_pct

,count,pct_of_total
total_rows,99441,100.0
null_purchase_at,0,0.0
null_approved_at,160,0.2
null_delivered_carrier_at,1783,1.8
null_delivered_customer_at,2965,3.0
null_estimated_delivery_at,0,0.0
null_avg_review_score,768,0.8
null_delivery_status,0,0.0


## Date range coverage

Confirms the order history window and that timestamps landed as real `TIMESTAMP` types, not strings.

In [4]:
q(f"""
SELECT
    MIN(order_purchase_at) AS earliest_order,
    MAX(order_purchase_at) AS latest_order,
    MIN(order_delivered_customer_at) AS earliest_delivery,
    MAX(order_delivered_customer_at) AS latest_delivery
FROM `{PROJECT_ID}.olist_warehouse.fact_orders`
""")

,earliest_order,latest_order,earliest_delivery,latest_delivery
0,2016-09-04 21:15:19+00:00,2018-10-17 17:30:18+00:00,2016-10-11 13:46:32+00:00,2018-10-17 13:22:46+00:00


## Quick sanity: delivery status distribution & review score

Cross-check against the business case doc's hypothesis (late deliveries -> lower reviews) before doing the deeper analysis in `04_satisfaction_analysis.ipynb`.

In [5]:
status_order = ["early", "on_time", "late_1_3_days", "late_4_7_days", "late_8_plus_days", "not_delivered"]
satisfaction = q(f"SELECT * FROM `{PROJECT_ID}.olist_reporting.mart_satisfaction_by_delivery`")
satisfaction["delivery_status"] = pd.Categorical(satisfaction["delivery_status"], categories=status_order, ordered=True)
satisfaction.sort_values("delivery_status").reset_index(drop=True)

,delivery_status,order_count,avg_review_score,avg_delivery_days,avg_delivery_delay_days
0,early,86718,4.296428,10.756114,-13.204175
1,on_time,1450,4.157931,16.909885,-0.273563
2,late_1_3_days,3132,3.595307,21.610100,1.742603
3,late_4_7_days,1748,2.105549,28.188215,6.189049
4,late_8_plus_days,2782,1.698059,44.323793,20.106593
5,not_delivered,2843,1.753254,NaN,NaN


## Assumption check: does a multi-seller order ship as one consolidated delivery?

`mart_transit_drivers` cuts transit by observable order shape. Before that model was
named honestly it claimed to classify *fulfilment mode*, resting on this inference:

> Olist records a single `order_delivered_carrier_date` and a single
> `order_delivered_customer_date` per order, so items from different sellers must have been
> merged into one shipment.

That reasoning is **invalid**. One delivery timestamp per
order is a fact about Olist's *schema*, not about how packages physically moved. A single
timestamp looks identical whether the items travelled together or arrived on three separate
days and one date got recorded.

The classification of orders by inferred fulfilment mode is risky as there is no significant evidence to support the fulfilment mode. The following cells try to explore whether customer reviews are spread out for multi-seller orders. The evidence is no conclusive. 

Instead of analysing fulfilment mode, the shipping performance can be broken down into orders with multi-sellers / single seller. 

In [10]:
# Is there a per-item delivery timestamp ANYWHERE in the dataset?
# The claim being tested is about all of Olist, so scan every raw table, not just orders.
schema = q(f"""
SELECT table_name, column_name, data_type
FROM `{PROJECT_ID}.olist_raw.INFORMATION_SCHEMA.COLUMNS`
WHERE NOT STARTS_WITH(table_name, '_dlt')
  AND (data_type IN ('TIMESTAMP', 'DATE', 'DATETIME')
       OR LOWER(column_name) LIKE '%date%'
       OR LOWER(column_name) LIKE '%_at%')
ORDER BY table_name, ordinal_position
""")

time_cols = schema[schema["data_type"].isin(["TIMESTAMP", "DATE", "DATETIME"])]
print(f"Every time-valued column across {time_cols.table_name.nunique()} raw tables:")
print(time_cols.to_string(index=False))

item_level = time_cols[time_cols["table_name"] == "order_items"]
print("\nTime columns in order_items (the only item-grain table):",
      list(item_level["column_name"]))
print("...of which actual delivery events:",
      [c for c in item_level["column_name"] if "deliver" in c.lower()] or "NONE")

Every time-valued column across 3 raw tables:
   table_name                   column_name data_type
  order_items           shipping_limit_date TIMESTAMP
order_reviews          review_creation_date TIMESTAMP
order_reviews       review_answer_timestamp TIMESTAMP
       orders      order_purchase_timestamp TIMESTAMP
       orders             order_approved_at TIMESTAMP
       orders  order_delivered_carrier_date TIMESTAMP
       orders order_delivered_customer_date TIMESTAMP
       orders order_estimated_delivery_date TIMESTAMP

Time columns in order_items (the only item-grain table): ['shipping_limit_date']
...of which actual delivery events: NONE


`order_items` carries no delivery timestamp at all — only `shipping_limit_date`. So the
direct test ("did these items arrive together?") **cannot be run on this dataset at any
level of effort.** The assumption is not merely unproven; it is unfalsifiable here.

What we *can* do is gather indirect evidence on whether the items plausibly moved as one
shipment.

In [11]:
# Indirect evidence 1: were the sellers even working to a common deadline, and is
# freight charged once per shipment or once per item?
# Restricted to 2+ item orders so basket size cannot drive the comparison.
q(f"""
WITH order_shape AS (
  SELECT
    order_id,
    COUNT(*) AS n_items,
    COUNT(DISTINCT seller_id) AS n_sellers,
    TIMESTAMP_DIFF(MAX(TIMESTAMP(shipping_limit_date)),
                   MIN(TIMESTAMP(shipping_limit_date)), HOUR) / 24.0 AS limit_spread_days,
    AVG(freight_value) AS freight_per_item
  FROM `{PROJECT_ID}.olist_raw.order_items`
  GROUP BY order_id
  HAVING COUNT(*) >= 2
)
SELECT
  IF(n_sellers > 1, 'multi-seller', 'single-seller') AS grp,
  COUNT(*) AS orders,
  ROUND(AVG(n_items), 2) AS avg_items,
  ROUND(APPROX_QUANTILES(limit_spread_days, 100)[OFFSET(50)], 2) AS median_limit_spread_days,
  ROUND(AVG(IF(limit_spread_days > 1, 1, 0)) * 100, 1) AS pct_orders_limits_over_1_day,
  ROUND(AVG(freight_per_item), 2) AS mean_freight_per_item
FROM order_shape
GROUP BY grp
ORDER BY grp
""")

,grp,orders,avg_items,median_limit_spread_days,pct_orders_limits_over_1_day,mean_freight_per_item
0,multi-seller,1278,2.43,0.0,19.5,19.57
1,single-seller,8525,2.43,0.0,0.0,18.43


In [12]:
# Indirect evidence 2: how often do the sellers on one order sit in different states?
# Merging those into a single physical shipment requires routing through a common hub.
q(f"""
WITH ms AS (
  SELECT
    i.order_id,
    COUNT(DISTINCT i.seller_id) AS n_sellers,
    COUNT(DISTINCT s.seller_state) AS n_seller_states
  FROM `{PROJECT_ID}.olist_raw.order_items` AS i
  JOIN `{PROJECT_ID}.olist_raw.sellers` AS s USING (seller_id)
  GROUP BY i.order_id
)
SELECT
  COUNTIF(n_sellers > 1) AS multi_seller_orders,
  COUNTIF(n_sellers > 1 AND n_seller_states > 1) AS sellers_in_different_states,
  ROUND(SAFE_DIVIDE(COUNTIF(n_sellers > 1 AND n_seller_states > 1),
                    COUNTIF(n_sellers > 1)) * 100, 1) AS pct_different_states
FROM ms
""")

,multi_seller_orders,sellers_in_different_states,pct_different_states
0,1278,510,39.9


Three things point **against** physical consolidation:

- **Freight is charged per item at the same rate** (~19.6 vs ~18.4) whether or not the order
  spans sellers. Genuinely merging items into one shipment should produce a freight economy;
  multi-seller orders instead pay slightly *more* per item — the signature of each seller
  shipping and charging independently.
- **19.5% of multi-seller orders have shipping deadlines more than a day apart**, against
  **0.0%** of single-seller orders. Sellers on the same order are working to different
  schedules, not a coordinated dispatch.
- **~40% of multi-seller orders have sellers in different states.** Merging those physically
  means routing every item through a common hub first.

So these look like **split shipments recorded under one timestamp**, not consolidated ones.
The test below asks what that does to the transit numbers.

In [13]:
# The decisive test: hold basket size AND distance constant, then compare.
# Multi-seller orders necessarily involve MORE shipments from MORE origins, so if the
# recorded date were the LAST arrival, they could not be faster than single-seller orders.
q(f"""
WITH shape AS (
  SELECT
    i.order_id,
    COUNT(*) AS n_items,
    COUNT(DISTINCT i.seller_id) AS n_sellers,
    COUNT(DISTINCT s.seller_state) AS n_seller_states,
    MIN(s.seller_state) AS min_seller_state
  FROM `{PROJECT_ID}.olist_raw.order_items` AS i
  JOIN `{PROJECT_ID}.olist_raw.sellers` AS s USING (seller_id)
  GROUP BY i.order_id
  HAVING COUNT(*) >= 2                      -- control for basket size
),
joined AS (
  SELECT
    sh.n_sellers > 1 AS is_multi,
    sh.n_items,
    NOT (sh.n_seller_states = 1
         AND sh.min_seller_state = c.customer_state) AS is_cross_state,
    TIMESTAMP_DIFF(o.order_delivered_customer_date,
                   o.order_delivered_carrier_date, HOUR) / 24.0 AS transit_days
  FROM shape AS sh
  JOIN `{PROJECT_ID}.olist_raw.orders` AS o USING (order_id)
  JOIN `{PROJECT_ID}.olist_raw.customers` AS c USING (customer_id)
  WHERE o.order_delivered_customer_date IS NOT NULL
    AND o.order_delivered_carrier_date IS NOT NULL
)
SELECT
  IF(is_cross_state, 'cross-state', 'same-state') AS distance_band,
  IF(is_multi, 'multi-seller', 'single-seller') AS grp,
  COUNT(*) AS orders,
  ROUND(AVG(n_items), 2) AS avg_items,
  ROUND(APPROX_QUANTILES(transit_days, 100)[OFFSET(50)], 2) AS median_transit_days,
  ROUND(AVG(transit_days), 2) AS mean_transit_days
FROM joined
GROUP BY distance_band, grp
ORDER BY distance_band, grp
""")

,distance_band,grp,orders,avg_items,median_transit_days,mean_transit_days
0,cross-state,multi-seller,934,2.41,6.88,7.83
1,cross-state,single-seller,5233,2.42,8.79,11.05
2,same-state,multi-seller,341,2.49,3.04,3.83
3,same-state,single-seller,3128,2.43,3.13,4.37


### Conclusion

Basket size is controlled (2.41 vs 2.42 items) and distance is controlled, yet cross-state
multi-seller orders record a **median transit of 6.88 days against 8.79** for single-seller
orders — roughly two days *faster*, while same-state orders are a wash (3.04 vs 3.13).

An order shipped from several sellers, ~40% of them in different states, cannot physically
reach the customer faster than the equivalent single-origin order. The most economical
explanation is that `order_delivered_customer_date` **does not represent the last item's
arrival** for multi-shipment orders, so their transit time is censored downward. The gap
appearing only cross-state fits: that is where individual shipment times diverge most.

**Consequences, carried into the model:**

1. The bucket cannot be called consolidation. It is renamed `multi_seller` —
   describing what is observed (several sellers) rather than what was assumed (one shipment).
2. Any finding that this group has faster transit is **not usable**. It is confounded with
   measurement censoring that we can detect but cannot correct without per-item delivery
   dates.
3. Answering the original consolidation-vs-direct-vs-dropship question properly needs data
   Olist does not contain: per-shipment tracking, carrier, or facility identifiers.

### One more possible fingerprint: the review survey

The full scan above turned up `order_reviews.review_creation_date`. Olist sends a
satisfaction survey **on delivery**, so an order that produced two surveys days apart is
weak circumstantial evidence of two delivery events.

In [14]:
q(f"""
WITH shape AS (
  SELECT order_id, COUNT(*) AS n_items, COUNT(DISTINCT seller_id) AS n_sellers
  FROM `{PROJECT_ID}.olist_raw.order_items` GROUP BY order_id
),
rev AS (
  SELECT order_id,
         COUNT(*) AS n_reviews,
         TIMESTAMP_DIFF(MAX(review_creation_date),
                        MIN(review_creation_date), HOUR) / 24.0 AS review_spread_days
  FROM `{PROJECT_ID}.olist_raw.order_reviews` GROUP BY order_id
)
SELECT
  CASE WHEN s.n_sellers > 1 THEN 'multi-seller'
       WHEN s.n_items >= 2  THEN 'single-seller 2+ items'
       ELSE 'single-seller 1 item' END AS grp,
  COUNT(*) AS orders,
  ROUND(AVG(IF(r.n_reviews > 1, 1, 0)) * 100, 2) AS pct_orders_with_multiple_reviews,
  ROUND(AVG(IF(r.n_reviews > 1 AND r.review_spread_days > 1, 1, 0)) * 100, 2) AS pct_reviews_over_1d_apart,
  ROUND(APPROX_QUANTILES(IF(r.n_reviews > 1, r.review_spread_days, NULL),
                         100)[OFFSET(50)], 2) AS median_spread_when_multiple
FROM shape AS s JOIN rev AS r USING (order_id)
GROUP BY grp ORDER BY grp
""")

,grp,orders,pct_orders_with_multiple_reviews,pct_reviews_over_1d_apart,median_spread_when_multiple
0,multi-seller,1264,1.03,0.79,3.0
1,single-seller 1 item,88227,0.53,0.31,3.0
2,single-seller 2+ items,8426,0.74,0.45,2.0


Multi-seller orders do carry multiple reviews about **twice** as often as single-item orders
(1.03% vs 0.53%), and twice as often with the surveys more than a day apart (0.79% vs 0.31%).
The direction is consistent with some multi-seller orders arriving in more than one delivery.

But the effect sits on a ~1% base — roughly **13 orders** out of 1,264 — and "complex orders
generate more complaints and re-surveys" explains the same pattern without any second
delivery. It is a hint, not a finding.
